<a href="https://colab.research.google.com/github/90splayer/Applied-ML/blob/main/Week6_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torchvision.datasets import Omniglot
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import random

transform = transforms.Compose([
    transforms.Resize((28,28)),
    transforms.ToTensor()
])

dataset = Omniglot(root='./data', background=True, download=True, transform=transform)

# Group images by class
class_dict = {}
for img, label in dataset:
    if label not in class_dict:
        class_dict[label] = []
    class_dict[label].append(img)

100%|██████████| 9.46M/9.46M [00:00<00:00, 157MB/s]


In [2]:
class FewShotDataset(Dataset):

    def __init__(self, class_dict, Nc=5, Ns=1, Nq=15):
        self.class_dict = class_dict
        self.classes = list(class_dict.keys())
        self.Nc = Nc
        self.Ns = Ns
        self.Nq = Nq

    def __len__(self):
        return 1000

    def __getitem__(self, idx):

        selected_classes = random.sample(self.classes, self.Nc)

        support_images = []
        support_labels = []
        query_images = []
        query_labels = []

        for i, c in enumerate(selected_classes):
            samples = random.sample(self.class_dict[c], self.Ns + self.Nq)

            support = samples[:self.Ns]
            query = samples[self.Ns:]

            support_images += support
            support_labels += [i] * self.Ns

            query_images += query
            query_labels += [i] * self.Nq

        support_images = torch.stack(support_images)
        query_images = torch.stack(query_images)

        support_labels = torch.tensor(support_labels)
        query_labels = torch.tensor(query_labels)

        return support_images, support_labels, query_images, query_labels

In [3]:
fewshot_dataset = FewShotDataset(class_dict, Nc=5, Ns=1, Nq=15)
loader = DataLoader(fewshot_dataset, batch_size=1)

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class ProtoNet(nn.Module):

    def __init__(self):
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )

        self.encoder = nn.Sequential(
            conv_block(1,64),
            conv_block(64,64),
            conv_block(64,64),
            conv_block(64,64)
        )

    def forward(self, x):
        x = self.encoder(x)
        return x.view(x.size(0), -1)

In [5]:
def euclidean_dist(x, y):

    n = x.size(0)
    m = y.size(0)

    x = x.unsqueeze(1).expand(n, m, -1)
    y = y.unsqueeze(0).expand(n, m, -1)

    return torch.pow(x - y, 2).sum(2)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ProtoNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Nc = 5
Ns = 1
Nq = 15

for episode in loader:

    support_x, support_y, query_x, query_y = episode

    support_x = support_x.squeeze(0).to(device)
    query_x = query_x.squeeze(0).to(device)

    support_y = support_y.squeeze(0).to(device)
    query_y = query_y.squeeze(0).to(device)

    support_embeddings = model(support_x)
    query_embeddings = model(query_x)

    prototypes = []

    for c in range(Nc):
        prototypes.append(
            support_embeddings[support_y == c].mean(0)
        )

    prototypes = torch.stack(prototypes)

    distances = euclidean_dist(query_embeddings, prototypes)

    log_p = F.log_softmax(-distances, dim=1)

    loss = F.nll_loss(log_p, query_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    pred = log_p.argmax(dim=1)
    acc = (pred == query_y).float().mean()

    print("Loss:", loss.item(), "Acc:", acc.item())

Loss: 2.7356607913970947 Acc: 0.6666666865348816
Loss: 2.1761772632598877 Acc: 0.6266666650772095
Loss: 3.0541067123413086 Acc: 0.6933333277702332
Loss: 2.2105941772460938 Acc: 0.7733333110809326
Loss: 6.133424282073975 Acc: 0.4933333396911621
Loss: 2.788986921310425 Acc: 0.5866666436195374
Loss: 0.9922341108322144 Acc: 0.8399999737739563
Loss: 1.3027966022491455 Acc: 0.8266666531562805
Loss: 1.3423354625701904 Acc: 0.7333333492279053
Loss: 3.129852771759033 Acc: 0.5600000023841858
Loss: 3.9015939235687256 Acc: 0.3866666555404663
Loss: 2.349283456802368 Acc: 0.5600000023841858
Loss: 2.788820743560791 Acc: 0.6399999856948853
Loss: 1.2803454399108887 Acc: 0.7200000286102295
Loss: 1.9108575582504272 Acc: 0.5600000023841858
Loss: 1.4219671487808228 Acc: 0.5733333230018616
Loss: 1.081235408782959 Acc: 0.8799999952316284
Loss: 4.171804904937744 Acc: 0.6933333277702332
Loss: 0.643337607383728 Acc: 0.8266666531562805
Loss: 1.1125937700271606 Acc: 0.7200000286102295
Loss: 0.7425177693367004 Acc